# Laboratorio 2: backpropagation y entrenamiento (EST852, Redes Neuronales II)

**Duración estimada de la parte práctica:** 70 min.

| Parte | Contenido | Min |
|:--|:--|:-:|
| 1 | Arquitectura y derivación (lectura) | 10 |
| 2 | Ejemplo numérico escalar (lectura y ejecución) | 5 |
| 3 | Implementar `backward()` en forma matricial | 15 |
| 4 | Verificación numérica del gradiente | 5 |
| 5 | Un paso manual frente a `partial_fit` de scikit-learn | 5 |
| 6 | Entrenamiento: tasa de aprendizaje, minibatch, inicialización | 10 |
| 7 | MLP frente a SVM en `load_digits` | 10 |
| 8 | Keras: la misma red con diferenciación automática | 10 |

**Cómo trabajar.** Las celdas con `...` son las que debes completar; después hay una verificación que muestra ✅ o ❌.

**Convención:** datos en columnas (*características × observaciones*), $W^{[l]}\in\mathbb{R}^{n_{l-1}\times n_l}$, igual que en la presentación.

In [ ]:
import numpy as np, sklearn, math
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
from sklearn.datasets import make_moons, load_digits
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

print('numpy', np.__version__, '| scikit-learn', sklearn.__version__)
np.set_printoptions(precision=4, suppress=True)

def verificar(nombre, condicion):
    try:
        ok = bool(condicion())
    except Exception:
        ok = False
    print(('✅ ' if ok else '❌ ') + nombre)

def sigmoide(z):
    return 1 / (1 + np.exp(-z))

## Parte 1. Arquitectura y derivación (10 min)

### 1.1 Arquitectura $2\to3\to1$

Una observación tiene 2 características; la capa oculta tiene 3 neuronas sigmoides; la salida es 1 neurona sigmoide. Parámetros: $W^{[1]}$ ($2\times3$), $b^{[1]}$ ($3\times1$), $W^{[2]}$ ($3\times1$), $b^{[2]}$ ($1\times1$); en total $6+3+3+1=13$.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 3.6)); ax.axis('off')
ins = [(0, .7), (0, .3)]; hid = [(1, .85), (1, .5), (1, .15)]
for (a_, b_) in ins:
    for (c_, d_) in hid: ax.plot([a_, c_], [b_, d_], c='gray', lw=1, zorder=1)
for (a_, b_) in hid: ax.plot([a_, 2], [b_, .5], c='gray', lw=1, zorder=1)
for i, (a_, b_) in enumerate(ins): ax.scatter(a_, b_, s=1200, c='#f28b82', zorder=2, edgecolor='k'); ax.text(a_, b_, f'$x_{i+1}$', ha='center', va='center', fontsize=12)
for i, (a_, b_) in enumerate(hid): ax.scatter(a_, b_, s=1200, c='#81c995', zorder=2, edgecolor='k'); ax.text(a_, b_, f'$a_{i+1}^{{[1]}}$', ha='center', va='center', fontsize=11)
ax.scatter(2, .5, s=1200, c='#8ab4f8', zorder=2, edgecolor='k'); ax.text(2, .5, '$a^{[2]}$', ha='center', va='center', fontsize=11)
ax.text(0, 1.0, 'Entrada\n$n=2$', ha='center'); ax.text(1, 1.05, 'Capa oculta\n3 neuronas, sigmoide', ha='center'); ax.text(2, .78, 'Salida\n1 neurona, sigmoide', ha='center')
ax.text(.5, .02, '$W^{[1]}$: $2\\times3$,  $b^{[1]}$: $3\\times1$', ha='center'); ax.text(1.5, .02, '$W^{[2]}$: $3\\times1$,  $b^{[2]}$: $1\\times1$', ha='center')
ax.set_xlim(-.3, 2.4); ax.set_ylim(-.05, 1.2); plt.tight_layout(); plt.show()

### 1.2 Propagación hacia adelante (una observación $\mathbf{x}=(x_1,x_2)^T$)

$$z^{[1]}_k=\sum_{j=1}^{2}w^{[1]}_{jk}x_j+b^{[1]}_k,\qquad a^{[1]}_k=\sigma\!\left(z^{[1]}_k\right),\quad k=1,2,3$$

$$z^{[2]}=\sum_{k=1}^{3}w^{[2]}_{k}\,a^{[1]}_k+b^{[2]},\qquad a^{[2]}=\sigma\!\left(z^{[2]}\right)$$

Pérdida de una observación (entropía cruzada binaria), con $a=a^{[2]}$:

$$\ell=-\bigl[y\log a+(1-y)\log(1-a)\bigr]$$

Con $m$ observaciones: $J=\frac1m\sum_{i=1}^{m}\ell^{(i)}$.

### 1.3 Derivadas parciales (una observación)

La regla de la cadena se aplica desde la pérdida hacia los parámetros:

$$\mathbf{x}\to z^{[1]}\to a^{[1]}\to z^{[2]}\to a^{[2]}\to \ell$$

**Paso 1: respecto de $z^{[2]}$.**

$$\frac{\partial\ell}{\partial a}=-\frac{y}{a}+\frac{1-y}{1-a}=\frac{a-y}{a(1-a)},\qquad\frac{\partial a}{\partial z^{[2]}}=a(1-a)\ \Rightarrow\ \delta^{[2]}:=\frac{\partial\ell}{\partial z^{[2]}}=a^{[2]}-y$$

**Paso 2: parámetros de la capa de salida.** Como $\partial z^{[2]}/\partial w^{[2]}_k=a^{[1]}_k$ y $\partial z^{[2]}/\partial b^{[2]}=1$:

$$\frac{\partial\ell}{\partial w^{[2]}_k}=\delta^{[2]}\,a^{[1]}_k,\qquad\frac{\partial\ell}{\partial b^{[2]}}=\delta^{[2]}$$

**Paso 3: error de la capa oculta.** Como $\partial z^{[2]}/\partial a^{[1]}_k=w^{[2]}_k$ y $\partial a^{[1]}_k/\partial z^{[1]}_k=a^{[1]}_k(1-a^{[1]}_k)$:

$$\delta^{[1]}_k:=\frac{\partial\ell}{\partial z^{[1]}_k}=\delta^{[2]}\,w^{[2]}_k\,a^{[1]}_k\left(1-a^{[1]}_k\right)$$

**Paso 4: parámetros de la capa oculta.** Como $\partial z^{[1]}_k/\partial w^{[1]}_{jk}=x_j$ y $\partial z^{[1]}_k/\partial b^{[1]}_k=1$:

$$\frac{\partial\ell}{\partial w^{[1]}_{jk}}=\delta^{[1]}_k\,x_j,\qquad\frac{\partial\ell}{\partial b^{[1]}_k}=\delta^{[1]}_k$$

### 1.4 Forma matricial ($m$ observaciones, $A^{[2]},Y\in\mathbb{R}^{1\times m}$)

| Gradiente | Fórmula | Forma |
|:--|:--|:-:|
| $dZ^{[2]}$ | $\frac1m\,(A^{[2]}-Y)$ | $1\times m$ |
| $dW^{[2]}$ | $A^{[1]}\,dZ^{[2]T}$ | $3\times1$ |
| $db^{[2]}$ | suma de las columnas de $dZ^{[2]}$ | $1\times1$ |
| $dA^{[1]}$ | $W^{[2]}\,dZ^{[2]}$ | $3\times m$ |
| $dZ^{[1]}$ | $dA^{[1]}\odot A^{[1]}\odot(1-A^{[1]})$ | $3\times m$ |
| $dW^{[1]}$ | $X\,dZ^{[1]T}$ | $2\times3$ |
| $db^{[1]}$ | suma de las columnas de $dZ^{[1]}$ | $3\times1$ |

$dV$ denota $\partial J/\partial V$ y $\odot$ el producto elemento a elemento. Cada gradiente tiene la **misma forma** que su parámetro.

## Parte 2. Ejemplo numérico, derivada por derivada (5 min)

Una observación $\mathbf{x}=(0{,}5;\,-1)^T$ con $y=1$. El código sigue **literalmente** las fórmulas de 1.2 y 1.3, con bucles y sin matrices.

In [ ]:
x = np.array([0.5, -1.0]); y = 1
W1 = np.array([[0.1, 0.2, -0.3],
               [0.4, -0.5, 0.6]])          # 2 x 3   (w1[j, k]: de la entrada j a la neurona oculta k)
b1 = np.array([0.1, 0.0, -0.1])            # 3
W2 = np.array([0.3, -0.2, 0.5])            # 3       (w2[k]: de la neurona oculta k a la salida)
b2 = 0.05

# Propagación hacia adelante (fórmulas de 1.2)
z1 = np.array([sum(W1[j, k] * x[j] for j in range(2)) + b1[k] for k in range(3)])
a1 = sigmoide(z1)
z2 = sum(W2[k] * a1[k] for k in range(3)) + b2
a2 = sigmoide(z2)
l  = -(y * np.log(a2) + (1 - y) * np.log(1 - a2))
print('z1 =', z1); print('a1 =', a1); print('z2 = %.4f   a2 = %.4f   pérdida = %.4f' % (z2, a2, l))

# Propagación hacia atrás (fórmulas de 1.3)
d2 = a2 - y                                                          # paso 1
dW2_s = np.array([d2 * a1[k] for k in range(3)]); db2_s = d2         # paso 2
d1 = np.array([d2 * W2[k] * a1[k] * (1 - a1[k]) for k in range(3)])  # paso 3
dW1_s = np.array([[d1[k] * x[j] for k in range(3)] for j in range(2)]) # paso 4
db1_s = d1.copy()
print('\ndelta2 =', round(d2, 4)); print('dW2 =', dW2_s); print('delta1 =', d1); print('dW1 =\n', dW1_s)

In [ ]:
# Comprobación independiente: diferencias finitas sobre cada w^{[1]}_{jk}
def perdida_escalar(W1, b1, W2, b2):
    a1 = sigmoide(W1.T @ x + b1); a2 = sigmoide(W2 @ a1 + b2)
    return -(y * np.log(a2) + (1 - y) * np.log(1 - a2))
eps = 1e-6; num = np.zeros_like(W1)
for j in range(2):
    for k in range(3):
        Wp = W1.copy(); Wp[j, k] += eps; Wm = W1.copy(); Wm[j, k] -= eps
        num[j, k] = (perdida_escalar(Wp, b1, W2, b2) - perdida_escalar(Wm, b1, W2, b2)) / (2 * eps)
print('máxima diferencia con el análisis (dW1):', np.abs(num - dW1_s).max())
verificar('Derivadas escalares coinciden con diferencias finitas', lambda: np.abs(num - dW1_s).max() < 1e-8)

## Parte 3. `backward()` en forma matricial (15 min)

`forward` y `costo` ya están escritos. En `backward` faltan **4 líneas** (marcadas con `...`): `dW2`, `dA1`, `dZ1` y `dW1`. Usa la tabla 1.4 y recuerda que cada gradiente debe tener la forma de su parámetro.

In [ ]:
def forward(X, P):
    Z1 = P['W1'].T @ X + P['b1']
    A1 = sigmoide(Z1)
    Z2 = P['W2'].T @ A1 + P['b2']
    A2 = sigmoide(Z2)
    return {'X': X, 'A1': A1, 'A2': A2}

def costo(A2, Y, eps=1e-12):
    A2 = np.clip(A2, eps, 1 - eps)
    return -np.mean(Y * np.log(A2) + (1 - Y) * np.log(1 - A2))

def backward(cache, P, Y):
    m = Y.shape[1]
    X, A1, A2 = cache['X'], cache['A1'], cache['A2']
    dZ2 = (A2 - Y) / m                                  # 1 x m
    db2 = dZ2.sum(axis=1, keepdims=True)                # 1 x 1
    dW2 = ...                                  # completar: 3 x 1
    dA1 = ...                               # completar: 3 x m
    dZ1 = ...                         # completar: 3 x m
    dW1 = ...                                   # completar: 2 x 3
    db1 = dZ1.sum(axis=1, keepdims=True)                # 3 x 1
    return {'W1': dW1, 'b1': db1, 'W2': dW2, 'b2': db2}

In [ ]:
# Mismo ejemplo de la Parte 2, ahora con matrices (m = 1)
P1 = {'W1': W1, 'b1': b1.reshape(3, 1), 'W2': W2.reshape(3, 1), 'b2': np.array([[b2]])}
X1 = x.reshape(2, 1); Y1 = np.array([[y]])
g1 = backward(forward(X1, P1), P1, Y1)

verificar('dW2 tiene forma (3, 1)', lambda: g1['W2'].shape == (3, 1))
verificar('dW1 tiene forma (2, 3)', lambda: g1['W1'].shape == (2, 3))
verificar('dW2 coincide con la versión escalar', lambda: np.allclose(g1['W2'].ravel(), dW2_s))
verificar('dW1 coincide con la versión escalar', lambda: np.allclose(g1['W1'], dW1_s))
verificar('db1 coincide con la versión escalar', lambda: np.allclose(g1['b1'].ravel(), db1_s))

## Parte 4. Verificación numérica del gradiente (5 min)

Para cada parámetro $\theta_k$, diferencias centrales:

$$\frac{\partial J}{\partial\theta_k}\approx\frac{J(\theta_k+\varepsilon)-J(\theta_k-\varepsilon)}{2\varepsilon}$$

**Ejercicio.** Completa el numerador y denominador en `gradiente_numerico`. La verificación usa $m=8$ observaciones aleatorias (varias observaciones, a diferencia de la Parte 3).

In [ ]:
def gradiente_numerico(P, X, Y, eps=1e-6):
    G = {}
    for k in P:
        G[k] = np.zeros_like(P[k])
        for idx in np.ndindex(*P[k].shape):
            original = P[k][idx]
            P[k][idx] = original + eps; J_mas = costo(forward(X, P)['A2'], Y)
            P[k][idx] = original - eps; J_menos = costo(forward(X, P)['A2'], Y)
            P[k][idx] = original
            G[k][idx] = ...
    return G

def error_relativo(g, gn):
    return np.linalg.norm(g - gn) / (np.linalg.norm(g) + np.linalg.norm(gn))

rng = np.random.default_rng(3)
Pg = {'W1': rng.normal(size=(2, 3)), 'b1': rng.normal(size=(3, 1)), 'W2': rng.normal(size=(3, 1)), 'b2': rng.normal(size=(1, 1))}
Xg = rng.normal(size=(2, 8)); Yg = (rng.random((1, 8)) > .5).astype(float)

def revisar():
    g = backward(forward(Xg, Pg), Pg, Yg); gn = gradiente_numerico(Pg, Xg, Yg)
    return {k: error_relativo(g[k], gn[k]) for k in Pg}

try:
    errores = revisar(); print({k: float('%.2e' % v) for k, v in errores.items()})
except Exception as e:
    errores = None; print('Aún no se puede calcular:', type(e).__name__)
verificar('Error relativo < 1e-6 en los cuatro bloques', lambda: max(errores.values()) < 1e-6)

**Pregunta 4.1.** Si introduces un error deliberado en `dZ1` (por ejemplo, olvidas el factor $(1-A^{[1]})$), ¿qué error relativo obtienes? Pruébalo y restaura el código.

## Parte 5. Un paso manual frente a `partial_fit` (5 min)

Un `MLPClassifier` realiza **exactamente** la actualización de descenso por gradiente si:

- `activation='logistic'`, `solver='sgd'`, `alpha=0` (sin penalización L2), `momentum=0`, `nesterovs_momentum=False`;
- `batch_size` igual al número de observaciones (gradiente completo).

En scikit-learn `coefs_[l]` tiene forma $n_{l-1}\times n_l$ (igual que $W^{[l]}$) y `X` es *observaciones × características*.

In [ ]:
rng = np.random.default_rng(0)
Xs = rng.normal(size=(40, 2)); ys = (Xs[:, 0] * Xs[:, 1] > 0).astype(int)
lr = 0.5
m_sk = MLPClassifier(hidden_layer_sizes=(3,), activation='logistic', solver='sgd', alpha=0,
                     momentum=0, nesterovs_momentum=False, batch_size=40,
                     learning_rate_init=lr, learning_rate='constant', random_state=1)
m_sk.partial_fit(Xs, ys, classes=[0, 1])             # inicializa las estructuras internas

# fijar pesos conocidos
Pk = {'W1': rng.normal(size=(2, 3)), 'b1': rng.normal(size=(3, 1)), 'W2': rng.normal(size=(3, 1)), 'b2': rng.normal(size=(1, 1))}
m_sk.coefs_ = [Pk['W1'].copy(), Pk['W2'].copy()]
m_sk.intercepts_ = [Pk['b1'].ravel().copy(), Pk['b2'].ravel().copy()]

# un paso con nuestro backward
g = backward(forward(Xs.T, Pk), Pk, ys.reshape(1, -1).astype(float))
manual = {k: Pk[k] - lr * g[k] for k in Pk}

m_sk.partial_fit(Xs, ys)                             # un paso de gradiente en scikit-learn
sk = {'W1': m_sk.coefs_[0], 'W2': m_sk.coefs_[1], 'b1': m_sk.intercepts_[0].reshape(-1, 1), 'b2': m_sk.intercepts_[1].reshape(-1, 1)}
for k in Pk: print(f'{k}: diferencia máxima = {np.abs(manual[k] - sk[k]).max():.2e}')
verificar('Un paso manual = un paso de scikit-learn', lambda: all(np.allclose(manual[k], sk[k], atol=1e-12) for k in Pk))

**Pregunta 5.1.** ¿Qué ocurre con la coincidencia si cambias `alpha=0` por `alpha=1`? ¿Y si usas `solver='adam'`? Pruébalo y explica por qué.

## Parte 6. Entrenamiento: tasa de aprendizaje, minibatch e inicialización (10 min)

Datos: *moons* (300 puntos, ruido 0,25); 210 de entrenamiento y 90 de prueba. Red $2\to4\to1$ propia.

**Ejercicio 6.1.** Completa la actualización de parámetros $\theta\leftarrow\theta-\eta\,\nabla_\theta J$.

In [ ]:
X, y = make_moons(300, noise=0.25, random_state=0)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
Xt, Yt = Xtr.T, ytr.reshape(1, -1).astype(float)
Xe, Ye = Xte.T, yte.reshape(1, -1).astype(float)

def inicializar(n_ent, n_oc, semilla=0, modo='aleatoria'):
    r = np.random.default_rng(semilla)
    if modo == 'ceros':
        W1, W2 = np.zeros((n_ent, n_oc)), np.zeros((n_oc, 1))
    else:
        W1, W2 = r.normal(0, .5, (n_ent, n_oc)), r.normal(0, .5, (n_oc, 1))
    return {'W1': W1, 'b1': np.zeros((n_oc, 1)), 'W2': W2, 'b2': np.zeros((1, 1))}

def entrenar(X, Y, n_oc=4, lr=1.0, epocas=300, batch=None, semilla=0, modo='aleatoria'):
    m = X.shape[1]; batch = batch or m
    P = inicializar(X.shape[0], n_oc, semilla, modo); r = np.random.default_rng(semilla + 1)
    hist = []; iteraciones = 0
    for ep in range(epocas):
        idx = r.permutation(m)
        for s in range(0, m, batch):
            b = idx[s:s + batch]
            g = backward(forward(X[:, b], P), P, Y[:, b])
            for k in P:
                P[k] = ...
            iteraciones += 1
        hist.append(costo(forward(X, P)['A2'], Y))
    return P, hist, iteraciones

def exactitud(P, X, Y):
    return float(((forward(X, P)['A2'] > .5) == Y).mean())

P_, h_, it_ = entrenar(Xt, Yt, lr=1.0, epocas=50)
verificar('El costo disminuye durante el entrenamiento', lambda: h_[-1] < h_[0])
verificar('50 épocas con batch completo = 50 iteraciones', lambda: it_ == 50)

**6.2 Tasa de aprendizaje.** 300 iteraciones de gradiente completo con distintos $\eta$.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9.5, 3.3))
print('tasa    J final   exactitud prueba (90 obs.)')
for lr_ in [0.01, 0.1, 1.0, 5.0]:
    P_, h_, _ = entrenar(Xt, Yt, 4, lr_, 300)
    print(f'{lr_:<6}  {h_[-1]:.3f}     {exactitud(P_, Xe, Ye):.3f}'); ax[0].plot(h_, label=f'tasa={lr_}')
ax[0].set_yscale('log'); ax[0].set_xlabel('época'); ax[0].set_ylabel('costo J'); ax[0].legend(fontsize=8); ax[0].set_title('Tasa de aprendizaje')

# 6.3 Minibatch: época vs iteración (tasa 0.5, 100 épocas)
print('\nbatch   it./época   it. totales   J final   exactitud prueba')
for b in [None, 32, 1]:
    P_, h_, it = entrenar(Xt, Yt, 4, 0.5, 100, b)
    m = Xt.shape[1]; esp = 100 * math.ceil(m / (b or m))
    print(f'{str(b or "completo"):<8}{math.ceil(m / (b or m)):<12}{it:<14}{h_[-1]:.3f}     {exactitud(P_, Xe, Ye):.3f}')
    ax[1].plot(h_, label=f'batch={b or "completo"}')
    verificar(f'iteraciones totales = épocas x ceil(m/b) para batch={b or "completo"}', lambda: it == esp)
ax[1].set_yscale('log'); ax[1].set_xlabel('época'); ax[1].legend(fontsize=8); ax[1].set_title('Minibatch')
plt.tight_layout(); plt.show()

**Preguntas 6.2 y 6.3.**

1. ¿Con qué tasa el costo no baja en 300 iteraciones? ¿Alguna tasa hace que el costo oscile o suba?
2. ¿Cuántas iteraciones por época hay con `batch=32` y $m=210$? Verifica con la tabla.
3. Con 90 observaciones de prueba, ¿cuánto vale una observación en puntos porcentuales?

**6.4 Inicialización.** Si todos los pesos parten en cero, las neuronas de la capa oculta reciben gradientes idénticos.

In [ ]:
Pz, hz, _ = entrenar(Xt, Yt, 4, 1.0, 300, modo='ceros')
Pr, hr, _ = entrenar(Xt, Yt, 4, 1.0, 300, modo='aleatoria')
print('Columnas de W1 (ceros) idénticas:', np.allclose(Pz['W1'][:, 0], Pz['W1'][:, 1]))
print(f'ceros:     J = {hz[-1]:.3f}   exactitud prueba = {exactitud(Pz, Xe, Ye):.3f}')
print(f'aleatoria: J = {hr[-1]:.3f}   exactitud prueba = {exactitud(Pr, Xe, Ye):.3f}')
verificar('Con ceros, las neuronas ocultas permanecen idénticas', lambda: np.allclose(Pz['W1'][:, 0], Pz['W1'][:, 3]))
verificar('Con ceros, J = ln 2 (predice 0.5 siempre)', lambda: abs(hz[-1] - math.log(2)) < 1e-6)

**Pregunta 6.4.** ¿Por qué con $W=0$ el gradiente de $W^{[1]}$ es exactamente 0 al comienzo? (Pista: $dA^{[1]}=W^{[2]}dZ^{[2]}$.)

## Parte 7. MLP frente a SVM en `load_digits` (10 min)

Mismo protocolo de la clase de SVM: partición estratificada 70/30, validación cruzada de 5 pliegues, escalado dentro del `Pipeline` y búsqueda en grilla.

In [ ]:
d = load_digits(); Xd, yd = d.data, d.target
Xdtr, Xdte, ydtr, ydte = train_test_split(Xd, yd, test_size=.3, random_state=0, stratify=yd)
cv = StratifiedKFold(5, shuffle=True, random_state=0)

svm = SVC(C=1, gamma=0.001)
cv_svm = cross_val_score(svm, Xdtr, ydtr, cv=cv).mean(); te_svm = svm.fit(Xdtr, ydtr).score(Xdte, ydte)

pipe = make_pipeline(StandardScaler(), MLPClassifier(max_iter=1000, random_state=0))
grilla = {'mlpclassifier__hidden_layer_sizes': [(16,), (64,), (128,)], 'mlpclassifier__alpha': [1e-4, 1e-2, 1.0]}
gs = GridSearchCV(pipe, grilla, cv=cv, n_jobs=-1).fit(Xdtr, ydtr)

n_te = len(ydte)
print(f'SVM (C=1, gamma=0.001):  CV = {cv_svm:.4f}   prueba = {te_svm:.4f}   errores = {round((1 - te_svm) * n_te)} de {n_te}')
print(f'MLP (mejor de 9):        CV = {gs.best_score_:.4f}   prueba = {gs.score(Xdte, ydte):.4f}   errores = {round((1 - gs.score(Xdte, ydte)) * n_te)} de {n_te}')
print('Mejores parámetros del MLP:', gs.best_params_)
print('\nGrilla (CV media):')
for p, s in zip(gs.cv_results_['params'], gs.cv_results_['mean_test_score']): print(f'  {p["mlpclassifier__hidden_layer_sizes"]!s:<8} alpha={p["mlpclassifier__alpha"]:<7} {s:.4f}')
print("\n¿MLPClassifier tiene class_weight?", 'class_weight' in MLPClassifier().get_params())

**Ejercicio 7.1 (detención temprana).** Entrena un `MLPClassifier((64,))` con `early_stopping=True`, `validation_fraction=0.15` y `n_iter_no_change=10`. Grafica `loss_curve_` y `validation_scores_`. ¿En qué época se detuvo? ¿Cuál fue la mejor accuracy de validación?

In [ ]:
es = make_pipeline(StandardScaler(), MLPClassifier((64,), max_iter=500, early_stopping=True,
                                                      validation_fraction=.15, n_iter_no_change=10, random_state=0)).fit(Xdtr, ydtr)
mm = es[-1]
print(f'épocas ejecutadas: {mm.n_iter_} | mejor accuracy de validación: {mm.best_validation_score_:.4f} | prueba: {es.score(Xdte, ydte):.4f}')
fig, ax = plt.subplots(1, 2, figsize=(9, 3.2))
ax[0].plot(mm.loss_curve_); ax[0].set_xlabel('época'); ax[0].set_ylabel('pérdida (entrenamiento)')
ax[1].plot(mm.validation_scores_); ax[1].axvline(np.argmax(mm.validation_scores_), ls='--', c='k'); ax[1].set_xlabel('época'); ax[1].set_ylabel('accuracy (validación)')
plt.tight_layout(); plt.show()

**Preguntas 7.2.**

1. La diferencia entre SVM y MLP en la prueba, ¿cuántas observaciones representa? ¿Es concluyente con una sola partición?
2. `MLPClassifier` no tiene `class_weight`. Si las clases estuvieran desbalanceadas, ¿qué alternativa de la clase de SVM usarías? (Pista: umbral sobre `predict_proba`.)
3. La SVM de esta comparación usa los píxeles sin escalar y el MLP los escala. ¿Cómo cambiarías la comparación para que sea justa?

## Parte 8. Keras: la misma red con diferenciación automática (10 min)

Aquí **no** se escribe backpropagation: el framework calcula los gradientes. Se usa la misma partición de *digits*, con píxeles divididos por 16.

In [ ]:
try:
    import os; os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
    import keras, tensorflow as tf
    print('TensorFlow', tf.__version__, '| Keras', keras.__version__)
    keras.utils.set_random_seed(0)
    Xk_tr, Xk_te = Xdtr / 16.0, Xdte / 16.0
    model = keras.Sequential([keras.Input((64,)),
                              keras.layers.Dense(64, activation='relu'),
                              keras.layers.Dense(10, activation='softmax')])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    print('parámetros:', model.count_params(), '= 64*64+64 + 64*10+10')
    hist = model.fit(Xk_tr, ydtr, epochs=100, batch_size=32, validation_split=.15,
                     callbacks=[keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)], verbose=0)
    loss, acc = model.evaluate(Xk_te, ydte, verbose=0)
    print(f'épocas: {len(hist.history["loss"])} | accuracy de prueba: {acc:.4f}')
    fig, ax = plt.subplots(1, 2, figsize=(9, 3.2))
    ax[0].plot(hist.history['loss'], label='entrenamiento'); ax[0].plot(hist.history['val_loss'], label='validación'); ax[0].set_xlabel('época'); ax[0].set_ylabel('entropía cruzada'); ax[0].legend()
    ax[1].plot(hist.history['accuracy'], label='entrenamiento'); ax[1].plot(hist.history['val_accuracy'], label='validación'); ax[1].set_xlabel('época'); ax[1].set_ylabel('accuracy'); ax[1].legend()
    plt.tight_layout(); plt.show()
except ImportError:
    print('TensorFlow/Keras no está instalado en este entorno. En Google Colab viene preinstalado.')

**Preguntas 8.1.**

1. ¿Qué función de activación usa la capa oculta aquí y cuál usamos en el Laboratorio? ¿Qué cambia en $\delta^{[1]}$?
2. Con softmax y entropía cruzada, el error de la capa de salida es $\hat y_k-y_k$. ¿Qué paralelo tiene con $\delta^{[2]}=a^{[2]}-y$?
3. ¿Cuántas épocas ejecutó el entrenamiento? Si llegó al máximo, ¿qué indica sobre `EarlyStopping`?

## Cierre

- $\delta^{[2]}=a^{[2]}-y$ se calcula una vez y se reutiliza en las derivadas de todos los parámetros anteriores.
- Cada gradiente tiene la forma de su parámetro; la verificación numérica comprueba la implementación.
- Un paso de `MLPClassifier` con la configuración adecuada es el gradiente completo que se implementó.
- El costo no es convexo: importan la inicialización, la tasa de aprendizaje y el criterio de parada.
- La selección de hiperparámetros se hace con validación cruzada, como en la clase de SVM.